<a href="https://colab.research.google.com/github/jyotidabass/Transfer-Learning-with-LLMs-Leveraging-Transfer-Learning-for-Domain-Adaptation-/blob/main/Transfer_Learning_with_LLMs_Leveraging_Transfer_Learning_for_Domain_Adaptation_and_Enhancing_Model_Performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install transformers
!pip install torch
!pip install sklearn

import pandas as pd
import torch
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

# Create dummy data
data = {
    'text': [
        'Patient has symptoms of fever and cough.',
        'Patient has symptoms of headache and fatigue.',
        'Patient has symptoms of chest pain and shortness of breath.',
        'Patient has symptoms of fever and headache.',
        'Patient has symptoms of cough and fatigue.',
        'Patient has symptoms of chest pain and headache.',
        'Patient has symptoms of shortness of breath and fatigue.',
        'Patient has symptoms of fever and chest pain.',
        'Patient has symptoms of cough and shortness of breath.',
        'Patient has symptoms of headache and chest pain.',
        'Patient has symptoms of fatigue and shortness of breath.',
        'Patient has symptoms of fever and fatigue.',
        'Patient has symptoms of cough and headache.',
        'Patient has symptoms of chest pain and fatigue.',
        'Patient has symptoms of shortness of breath and headache.'
    ],
    'label': [1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]  # 1 for disease, 0 for no disease
}

# Load the pre-trained BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Split the data into training and validation sets
train_text, val_text, train_labels, val_labels = train_test_split(
    data['text'], data['label'], random_state=42, test_size=0.2, stratify=data['label']
)

# Create a custom dataset class
class MedicalDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer.encode_plus(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Create dataset and data loader for training and validation
max_len = 512
batch_size = 16

train_dataset = MedicalDataset(train_text, train_labels, tokenizer, max_len)
val_dataset = MedicalDataset(val_text, val_labels, tokenizer, max_len)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

# Fine-tune the pre-trained BERT model
class MedicalDiagnosisModel(nn.Module):
    def __init__(self):
        super(MedicalDiagnosisModel, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.drop = nn.Dropout(p=0.3)
        self.out = nn.Linear(self.bert.config.hidden_size, 2)  # 2 classes for medical diagnosis

    def forward(self, input_ids, attention_mask):
        # Get the 'last_hidden_state' and apply mean pooling
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        pooled_output = output.mean(dim=1)  # Mean pooling

        output = self.drop(pooled_output)
        return self.out(output)

# Initialize the model, optimizer, and loss function
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MedicalDiagnosisModel()
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5)

# Train the model
for epoch in range(5):  # 5 epochs
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {total_loss / len(train_loader)}')

# Evaluate the fine-tuned model
model.eval()
total_correct = 0
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids, attention_mask)
        _, predicted = torch.max(outputs, dim=1)
        total_correct += (predicted == labels).sum().item()

accuracy = total_correct / len(val_labels)
print(f'Validation Accuracy: {accuracy:.4f}')

# Evaluate the model using domain-specific metrics
predictions = []
labels = []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids, attention_mask)
        _, predicted = torch.max(outputs, dim=1)
        predictions.extend(predicted.cpu().numpy())
        labels.extend(batch['labels'].cpu().numpy())

precision = precision_score(labels, predictions)
recall = recall_score(labels, predictions)

print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')

  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Epoch 1, Loss: 0.7541854977607727
Epoch 2, Loss: 0.6749207377433777
Epoch 3, Loss: 0.7184168696403503
Epoch 4, Loss: 0.6747180819511414
Epoch 5, Loss: 0.6522006988525391
Validation Accuracy: 0.6667
Precision: 1.0000
Recall: 0.5000
